## **1. DRIVE AND PATHS SETUP**

In [ ]:
from google.colab import drive
import os
import sys

drive.mount('/content/drive')
project_folder = "/content/drive/MyDrive/FAIMDL Project"
repo_name = "MaskArchitectureAnomaly_CourseProject"

repo_path = os.path.join(project_folder, repo_name)
eval_folder = os.path.join(repo_path, "eval")

%cd {eval_folder}


if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
if eval_folder not in sys.path:
    sys.path.insert(0, eval_folder)

print(f" Current location: {eval_folder}")

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/10fnUuRt9u3uQO-g19MaijePSMIFVeM0v/FAIMDL Project/MaskArchitectureAnomaly_CourseProject/eval
 Current location: /content/drive/MyDrive/FAIMDL Project/MaskArchitectureAnomaly_CourseProject/eval


## **2. LIBRARIES INSTALLATION**

In [ ]:
# The --no-deps flag prevents the library from downgrading/corrupting Colab's default packages
!pip install ood-metrics --no-deps -q

## **3. IMPORTS AND TRANSFORMS SETUP**


In [ ]:
import glob
import torch
import random
from PIL import Image
import numpy as np
import os.path as osp

# Import the ERFNet model and Out-of-Distribution metrics
from erfnet import ERFNet
from ood_metrics import fpr_at_95_tpr, calc_metrics, plot_roc, plot_pr, plot_barcode
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

# Set seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

NUM_CHANNELS = 3
NUM_CLASSES = 20 # 19 standard Cityscapes classes + 1 ("Anomaly" class)

# Transforms to adapt input images for ERFNet
input_transform = Compose([
    Resize((512, 1024), Image.BILINEAR),
    ToTensor(),
])

# Transforms for Ground Truth masks (using NEAREST to avoid blurring class boundaries)
target_transform = Compose([
    Resize((512, 1024), Image.NEAREST),
])

In [ ]:
# EXTRACTION UTILITY

# 1. Navigate to your dataset folder
%cd "/content/drive/MyDrive/FAIMDL Project/Anomaly_Validation_Datasets"

# 2. Extract the zip file

!unzip -q -n Anomaly_Validation_Datasets.zip



/content/drive/.shortcut-targets-by-id/10fnUuRt9u3uQO-g19MaijePSMIFVeM0v/FAIMDL Project/Anomaly_Validation_Datasets
unzip:  cannot find or open Anomaly_Validation_Datasets.zip, Anomaly_Validation_Datasets.zip.zip or Anomaly_Validation_Datasets.zip.ZIP.


## **4. CONFIGURATION AND MODEL INITIALIZATION**

In [ ]:
import argparse

config_parser = argparse.ArgumentParser(description="Anomaly Evaluation Setup")

# Path to the validation images directory.
val_images_path = "/content/drive/MyDrive/FAIMDL Project/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/*.png"
config_parser.add_argument("--input", default=val_images_path, nargs="+", help="Input images path (can be a glob pattern)")
config_parser.add_argument('--loadDir', default="/content/drive/MyDrive/FAIMDL Project/MaskArchitectureAnomaly_CourseProject/trained_models/")
config_parser.add_argument('--loadWeights', default="erfnet_pretrained.pth")
config_parser.add_argument('--loadModel', default="erfnet.py")
config_parser.add_argument('--subset', default="val")
config_parser.add_argument('--num-workers', type=int, default=4)
config_parser.add_argument('--batch-size', type=int, default=1)
config_parser.add_argument('--cpu', action='store_true')

# we pass an empty list to prevent jupyter env crashes
parsed_args = config_parser.parse_args(args=[])

# Data structures to store evaluation metrics for different Post-Hoc approaches
scores_maxlogit = []
scores_msp = []
scores_maxentropy = []

# Array to store the actual anomaly masks (Ground Truth)
ground_truths = []

# Safely initialize the text file to log quantitative results
results_file = 'results.txt'
if not os.path.exists(results_file):
    with open(results_file, 'w') as f:
        pass

# Securely build paths using os.path.join to prevent cross-platform slashes issues
path_to_model = os.path.join(parsed_args.loadDir, parsed_args.loadModel)
path_to_weights = os.path.join(parsed_args.loadDir, parsed_args.loadWeights)

print(f"-> Architecture script: {path_to_model}")
print(f"-> Pre-trained weights: {path_to_weights}")

# Instantiate the ERFNet architecture (Standard 19 classes + 1 Anomaly class)
anomaly_model = ERFNet(NUM_CLASSES)

# Offload the model to GPU if available
if not parsed_args.cpu:
    anomaly_model = torch.nn.DataParallel(anomaly_model).cuda()

def inject_weights_safely(network, checkpoint_dict):
    """
    Safely injects weights into the network, handling potential 'module.' prefixes
    that are often generated when saving models wrapped in DataParallel.
    """
    current_state = network.state_dict()
    for key, tensor_val in checkpoint_dict.items():
        if key not in current_state:
            # Strip the DataParallel prefix if present
            if key.startswith("module."):
                stripped_key = key.split("module.")[-1]
                current_state[stripped_key].copy_(tensor_val)
            else:
                print(f"[Warning] Layer '{key}' ignored during loading.")
                continue
        else:
            current_state[key].copy_(tensor_val)
    return network

# Extract the dictionary from the saved checkpoint
loaded_dict = torch.load(path_to_weights, map_location=lambda storage, loc: storage)

# Apply the weights to our model
anomaly_model = inject_weights_safely(anomaly_model, loaded_dict)
print(" ERFNet successfully initialized")

# Lock the network in evaluation mode
anomaly_model.eval()

-> Architecture script: /content/drive/MyDrive/FAIMDL Project/MaskArchitectureAnomaly_CourseProject/trained_models/erfnet.py
-> Pre-trained weights: /content/drive/MyDrive/FAIMDL Project/MaskArchitectureAnomaly_CourseProject/trained_models/erfnet_pretrained.pth
 ERFNet successfully initialized


DataParallel(
  (module): ERFNet(
    (encoder): Encoder(
      (initial_block): DownsamplerBlock(
        (conv): Conv2d(3, 13, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      )
      (layers): ModuleList(
        (0): DownsamplerBlock(
          (conv): Conv2d(16, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1-5): 5 x non_bottleneck_1d(
          (conv3x1_1): Conv2d(64, 64, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
          (conv1x3_1): Conv2d(64, 64, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
          (bn1): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True

# **DEBUG**

In [ ]:
import numpy as np
from PIL import Image
import glob

# Mask path (Ground Truth)
gt_dir = "/content/drive/MyDrive/FAIMDL Project/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/labels_masks"

# First mask
gt_files = glob.glob(f"{gt_dir}/*.png")

if len(gt_files) > 0:
    test_mask = np.array(Image.open(gt_files[0]))
    valori_unici = np.unique(test_mask)
    print(f"Unique values found: {valori_unici}")
else:
    print("Zero mask found")

Unique values found: [  0   1 255]


## **5. INFERENCE LOOP AND POST-HOC CALCULATION**

In [ ]:
import torch.nn.functional as F
import os
import glob
from PIL import Image
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import average_precision_score
import gc

IGNORE_INDEX = 255


ANOMALY_DATASETS = {
    "FS_LostFound_full": "/content/drive/MyDrive/FAIMDL Project/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full",
    "RoadAnomaly": "/content/drive/MyDrive/FAIMDL Project/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly",
    "RoadAnomaly21": "/content/drive/MyDrive/FAIMDL Project/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly21",
    "fs_static": "/content/drive/MyDrive/FAIMDL Project/Anomaly_Validation_Datasets/Validation_Dataset/fs_static",
    "Road Obstacle": "/content/drive/MyDrive/FAIMDL Project/Anomaly_Validation_Datasets/Validation_Dataset/RoadObsticle21",
}

print("STEP 7: ANOMALY DETECTION ON ALL DATASETS")

# START OF THE OUTER LOOP
for dataset_name, dataset_path in ANOMALY_DATASETS.items():
    print(f"\n Dataset: {dataset_name}")
    print("-" * 90)

    img_dir = os.path.join(dataset_path, "images")

    if not os.path.exists(img_dir):
        print(f"Path not found: {img_dir}")
        continue


    img_files = []
    for ext in ['*.png', '*.jpg', '*.jpeg', '*.webp']:
        img_files.extend(glob.glob(os.path.join(img_dir, ext)))

    if len(img_files) == 0:
        print(f"No images found in {img_dir}")
        continue

    print(f"Found {len(img_files)} images")

    filtered_ground_truths = []
    filtered_scores_maxlogit = []
    filtered_scores_msp = []
    filtered_scores_maxentropy = []

    for img_path in tqdm(img_files, desc=f"   Processing {dataset_name}", leave=False):

        raw_img = Image.open(img_path).convert('RGB')
        input_tensor = input_transform(raw_img).unsqueeze(0).float().cuda()

        with torch.no_grad():
            logits = anomaly_model(input_tensor)

        squeezed_logits = logits.squeeze(0)
        probs = F.softmax(squeezed_logits, dim=0)

        logits_np = squeezed_logits.data.cpu().numpy()
        score_maxlogit = 1.0 - np.max(logits_np, axis=0)

        max_probs = torch.max(probs, dim=0)[0]
        score_msp = 1.0 - max_probs.data.cpu().numpy()

        eps = 1e-12
        entropy_tensor = -torch.sum(probs * torch.log(probs + eps), dim=0)
        score_entropy = entropy_tensor.data.cpu().numpy()

        gt_path = img_path.replace("images", "labels_masks").rsplit(".", 1)[0] + ".png"

        if not os.path.exists(gt_path):

             gt_path = gt_path.replace(".png", ".webp")
             if not os.path.exists(gt_path):
                 continue

        gt_image = target_transform(Image.open(gt_path))
        gt_numpy = np.array(gt_image)


        if "RoadAnomaly" in dataset_name or "Road Obstacle" in dataset_name:
            gt_numpy = np.where((gt_numpy == 2), 1, gt_numpy)


        valid_mask = (gt_numpy != IGNORE_INDEX) & ((gt_numpy == 0) | (gt_numpy == 1))

        if 1 not in gt_numpy[valid_mask]:
            continue

        filtered_ground_truths.append(gt_numpy[valid_mask])
        filtered_scores_maxlogit.append(score_maxlogit[valid_mask])
        filtered_scores_msp.append(score_msp[valid_mask])
        filtered_scores_maxentropy.append(score_entropy[valid_mask])

        del input_tensor, logits, squeezed_logits, probs, max_probs, entropy_tensor
        del logits_np, score_maxlogit, score_msp, score_entropy, gt_numpy, gt_image
        torch.cuda.empty_cache()

    if len(filtered_ground_truths) == 0:
        print(f"No valid samples with anomalies found in {dataset_name}")
        continue

    flat_gts = np.concatenate(filtered_ground_truths)
    flat_maxlogit = np.concatenate(filtered_scores_maxlogit)
    flat_msp = np.concatenate(filtered_scores_msp)
    flat_maxentropy = np.concatenate(filtered_scores_maxentropy)

    evaluated_metrics = {
        "MSP": flat_msp,
        "MaxLogit": flat_maxlogit,
        "MaxEntropy": flat_maxentropy
    }

    print(f"\n Results for {dataset_name}:")
    print(f"   {'-'*70}")

    with open(results_file, 'a') as f:
        f.write(f"\n--- {dataset_name} ---\n")

        for metric_name, flat_scores in evaluated_metrics.items():
            try:
                prc_auc = average_precision_score(flat_gts, flat_scores) * 100.0
                fpr95 = fpr_at_95_tpr(flat_scores, flat_gts) * 100.0
            except Exception:
                prc_auc, fpr95 = 0.0, 100.0

            print(f"     {metric_name:12} -> AUPRC: {prc_auc:6.2f}%  |  FPR@TPR95: {fpr95:6.2f}%")
            f.write(f"{metric_name} - AUPRC: {prc_auc:.2f}% | FPR@TPR95: {fpr95:.2f}%\n")

        f.write("========================\n")

    print(f"   {'-'*70}\n")

    del filtered_ground_truths, filtered_scores_maxlogit, filtered_scores_msp, filtered_scores_maxentropy
    del flat_gts, flat_maxlogit, flat_msp, flat_maxentropy
    gc.collect()

print(f"Results appended to: {results_file}")

STEP 7: ANOMALY DETECTION ON ALL DATASETS

 Dataset: FS_LostFound_full
------------------------------------------------------------------------------------------
Found 100 images



 Results for FS_LostFound_full:
   ----------------------------------------------------------------------
     MSP          -> AUPRC:   1.75%  |  FPR@TPR95:  50.59%
     MaxLogit     -> AUPRC:   3.30%  |  FPR@TPR95:  45.49%
     MaxEntropy   -> AUPRC:   2.58%  |  FPR@TPR95:  50.16%
   ----------------------------------------------------------------------


 Dataset: RoadAnomaly
------------------------------------------------------------------------------------------
Found 60 images



 Results for RoadAnomaly:
   ----------------------------------------------------------------------
     MSP          -> AUPRC:  12.42%  |  FPR@TPR95:  82.58%
     MaxLogit     -> AUPRC:  15.58%  |  FPR@TPR95:  73.25%
     MaxEntropy   -> AUPRC:  12.67%  |  FPR@TPR95:  82.75%
   ----------------------------------------------------------------------


 Dataset: RoadAnomaly21
------------------------------------------------------------------------------------------
Found 10 images



 Results for RoadAnomaly21:
   ----------------------------------------------------------------------
     MSP          -> AUPRC:  29.10%  |  FPR@TPR95:  62.55%
     MaxLogit     -> AUPRC:  38.32%  |  FPR@TPR95:  59.34%
     MaxEntropy   -> AUPRC:  30.97%  |  FPR@TPR95:  62.66%
   ----------------------------------------------------------------------


 Dataset: fs_static
------------------------------------------------------------------------------------------
Found 30 images



 Results for fs_static:
   ----------------------------------------------------------------------
     MSP          -> AUPRC:   7.47%  |  FPR@TPR95:  41.84%
     MaxLogit     -> AUPRC:   9.50%  |  FPR@TPR95:  40.30%
     MaxEntropy   -> AUPRC:   8.84%  |  FPR@TPR95:  41.55%
   ----------------------------------------------------------------------


 Dataset: Road Obstacle
------------------------------------------------------------------------------------------
Found 30 images



 Results for Road Obstacle:
   ----------------------------------------------------------------------
     MSP          -> AUPRC:   2.71%  |  FPR@TPR95:  65.22%
     MaxLogit     -> AUPRC:   4.63%  |  FPR@TPR95:  48.44%
     MaxEntropy   -> AUPRC:   3.04%  |  FPR@TPR95:  65.91%
   ----------------------------------------------------------------------

Results appended to: results.txt


# **CODE for mIoU**

In [ ]:
# 7.1 CITYSCAPES DATASET PREPARATION FOR mIoU

import os
import shutil
import glob

local_data_dir = "/content/cityscapes_eval"
os.makedirs(local_data_dir, exist_ok=True)

drive_source_dir = "/content/drive/MyDrive/FAIMDL Project/Cityscapes_Dataset"
images_zip_drive = os.path.join(drive_source_dir, "leftImg8bit_trainvaltest.zip")
labels_zip_drive = os.path.join(drive_source_dir, "gtFine_trainvaltest.zip")


# Copy files locally first to prevent unzip issues with shortcuts
local_images_zip = "/content/leftImg8bit.zip"
local_labels_zip = "/content/gtFine.zip"

if os.path.exists(images_zip_drive) and os.path.exists(labels_zip_drive):
    if not os.path.exists(local_images_zip):
        shutil.copy2(images_zip_drive, local_images_zip)
    if not os.path.exists(local_labels_zip):
        shutil.copy2(labels_zip_drive, local_labels_zip)

    # Extract the files locally
    !unzip -q -o "{local_images_zip}" -d "{local_data_dir}"
    !unzip -q -o "{local_labels_zip}" -d "{local_data_dir}"

    # Delite local zip
    os.remove(local_images_zip)
    os.remove(local_labels_zip)

    # Final check
    png_count = len(glob.glob(f"{local_data_dir}/**/*.png", recursive=True))
    print(f"Found {png_count} .png files.")

    if png_count == 0:
        print("ERROR: Still no images extracted.")
else:
    print("Zip files not detected at the specified Drive path.")

Found 20000 .png files.


In [ ]:


# Install the toolkit quietly
!pip install cityscapesscripts -q

# Set the mandatory environment variable required by the cityscapes-scripts pipeline
os.environ['CITYSCAPES_DATASET'] = local_data_dir


# Invoke the official CLI tool to map the 34 original semantic IDs to the 19 evaluation IDs
!csCreateTrainIdLabelImgs
print("Label preparation complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 9.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.6/473.6 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.1 MB/s eta 0:00:00
Processing 5000 annotation files
Progress: 100.0 % Label preparation complete


In [ ]:
# 7.3 MIOU EVALUATION SETUP & TRANSFORMS

import argparse
from torchvision.transforms import Compose, Resize, ToTensor
from dataset import cityscapes
from transform import Relabel, ToLabel
from iouEval import iouEval

# 1. Define Image and Target Transformations for Cityscapes
# We resize to 512 height to match ERFNet's expected input resolution during evaluation
val_input_transforms = Compose([
    Resize(512, Image.BILINEAR),
    ToTensor(),
])

# Targets need NEAREST interpolation to preserve exact class IDs (no blurring).
# Class 255 (ignored/void in Cityscapes) is mapped to 19 to keep tensor indices continuous.
val_target_transforms = Compose([
    Resize(512, Image.NEAREST),
    ToLabel(),
    Relabel(255, 19),
])

# 2. Configuration Parameters for the Standard Evaluation Run
miou_config_parser = argparse.ArgumentParser(description="Cityscapes mIoU Evaluation Setup")

# Pointing securely to our Google Drive directory for the pre-trained weights
miou_config_parser.add_argument('--loadDir', default="/content/drive/MyDrive/FAIMDL Project/MaskArchitectureAnomaly_CourseProject/trained_models/")
miou_config_parser.add_argument('--loadWeights', default="erfnet_pretrained.pth")
miou_config_parser.add_argument('--loadModel', default="erfnet.py")
miou_config_parser.add_argument('--subset', default="val")
miou_config_parser.add_argument('--datadir', default="/content/cityscapes_eval") # Matches our local extraction folder
miou_config_parser.add_argument('--num-workers', type=int, default=2)
miou_config_parser.add_argument('--batch-size', type=int, default=1)
miou_config_parser.add_argument('--cpu', action='store_true')


miou_args = miou_config_parser.parse_args(args=[])

# 3. Sanity Check for the Dataset Directory
if not os.path.exists(miou_args.datadir):
    print(f"CRITICAL ERROR: The dataset directory '{miou_args.datadir}' does not exist.")
else:
    print(f"Directory found: {miou_args.datadir}. Ready to load the Cityscapes Validation Dataset!")

Directory found: /content/cityscapes_eval. Ready to load the Cityscapes Validation Dataset!


In [ ]:

from torch.utils.data import DataLoader
from tqdm import tqdm
from iouEval import getColorEntry

# 7.4 RUNNING THE mIoU EVALUATION LOOP

# 1. Initialize the DataLoader using our secure configurations
val_dataset = cityscapes(miou_args.datadir, val_input_transforms, val_target_transforms, subset=miou_args.subset)

if len(val_dataset) == 0:
    print("\n CRITICAL ERROR: DataLoader do not found images.")
else:
    val_dataloader = DataLoader(val_dataset, num_workers=miou_args.num_workers, batch_size=miou_args.batch_size, shuffle=False)

    # (19 Cityscapes classes + 1 Void/Anomaly class mapped to ID 19)
    evaluator = iouEval(20)




    # 3. Evaluation Loop
    for step, (batch_imgs, batch_labels, _, _) in enumerate(tqdm(val_dataloader, desc="Evaluating Cityscapes")):

        if not miou_args.cpu:
            batch_imgs = batch_imgs.cuda()
            batch_labels = batch_labels.cuda()

        with torch.no_grad():
            outputs = anomaly_model(batch_imgs)

        # Extract predictions and accumulate in the confusion matrix
        predictions = outputs.max(1)[1].unsqueeze(1).data
        evaluator.addBatch(predictions, batch_labels)

    # 4. Fetch final quantitative results
    mean_iou, per_class_iou = evaluator.getIoU()

    # 5. Professional formatting for the final report
    cityscapes_classes = [
        "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light",
        "traffic sign", "vegetation", "terrain", "sky", "person", "rider", "car",
        "truck", "bus", "train", "motorcycle", "bicycle"
    ]

    print("PER-CLASS INTERSECTION OVER UNION (IoU)")

    # Iterate strictly over the 19 standard classes to print results
    for i, class_name in enumerate(cityscapes_classes):
        iou_val = per_class_iou[i]
        color_code = getColorEntry(iou_val)
        formatted_iou = f"{iou_val*100:6.2f}%"
        print(f"{color_code}{class_name:15}: {formatted_iou}\033[0m")

    print("-" * 45)


    # Final overarching mIoU score (calculated on all classes managed by iouEval)
    final_color = getColorEntry(mean_iou)
    print(f" MEAN IoU (mIoU)  : {final_color}{mean_iou*100:.2f}%\033[0m")
    print("="*45)

/content/cityscapes_eval/leftImg8bit/val /content/cityscapes_eval/gtFine/val


Evaluating Cityscapes: 100%|██████████| 500/500 [01:20<00:00,  6.24it/s]

PER-CLASS INTERSECTION OVER UNION (IoU)
road           :  97.62%
sidewalk       :  81.37%
building       :  90.77%
wall           :  49.43%
fence          :  54.93%
pole           :  60.81%
traffic light  :  62.60%
traffic sign   :  72.32%
vegetation     :  91.35%
terrain        :  60.97%
sky            :  93.38%
person         :  76.11%
rider          :  53.45%
car            :  92.91%
truck          :  72.78%
bus            :  78.87%
train          :  63.86%
motorcycle     :  46.41%
bicycle        :  71.89%
---------------------------------------------
 MEAN IoU (mIoU)  : 72.20%
